# M1 Hard Normal 오탐 분석

## tl;dr

회사 제공 `normal` 라벨은 유지한다. 이 노트북은 `compact13 + threshold 0.6` 기준에서
정상인데 위험처럼 잡힌 normal 이벤트를 `hard_normal`로 audit하고, 다음 해석/학습에서
주의할 normal 패턴을 정리한다.

## Context & Methods

- 입력: 10번 leakage-safe prediction/audit, 08번 feature expansion feature, 04번 label/window audit
- 기준: `normal` vs `efd_possible/pre_event`
- main dataset: `strict_no_event20`
- sensitivity dataset: `strict_no_event20_no_event67`
- feature: `compact13`
- threshold: `0.6`

### Key Assumptions

- 회사 제공 normal 라벨은 정답 기준으로 유지한다.
- normal을 삭제하거나 재라벨링하지 않는다.
- 이번 단계는 모델 고도화가 아니라 normal 오탐 원인과 hard normal 후보 문서화다.

In [1]:
from __future__ import annotations

from pathlib import Path
import math

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUTPUT_DIR = Path("07_데이터산출물")
PREDICTIONS_PATH = OUTPUT_DIR / "m1_compact13_leakage_safe_cv_predictions.csv"
MISCLASSIFICATION_PATH = OUTPUT_DIR / "m1_compact13_leakage_safe_misclassification_audit.csv"
FEATURES_PATH = OUTPUT_DIR / "m1_feature_expansion_features.csv"
LABEL_WINDOW_AUDIT_PATH = OUTPUT_DIR / "m1_label_window_audit.csv"
FEATURE_SET_SUMMARY_PATH = OUTPUT_DIR / "m1_compact_feature_set_summary.csv"
REPORT_PATH = OUTPUT_DIR / "11_M1_hard_normal_false_positive_보고서.md"

MAIN_DATASET = "strict_no_event20"
SENSITIVITY_DATASET = "strict_no_event20_no_event67"
THRESHOLD = 0.6
MODEL_NAME = "logistic_balanced"
STRATEGIES = ["expanded154", "locked_compact13", "fold_selected_compact13"]
COMPACT_STRATEGIES = ["locked_compact13", "fold_selected_compact13"]

OUTPUT_FILES = {
    "audit": OUTPUT_DIR / "m1_hard_normal_audit.csv",
    "feature_profile": OUTPUT_DIR / "m1_hard_normal_feature_profile.csv",
    "substation_summary": OUTPUT_DIR / "m1_hard_normal_substation_summary.csv",
    "decision_notes": OUTPUT_DIR / "m1_hard_normal_decision_notes.csv",
    "probability_ranking_png": OUTPUT_DIR / "m1_hard_normal_probability_ranking.png",
    "feature_profile_png": OUTPUT_DIR / "m1_hard_normal_feature_profile.png",
    "substation_distribution_png": OUTPUT_DIR / "m1_hard_normal_substation_distribution.png",
}

print("inputs")
for path in [
    PREDICTIONS_PATH,
    MISCLASSIFICATION_PATH,
    FEATURES_PATH,
    LABEL_WINDOW_AUDIT_PATH,
    FEATURE_SET_SUMMARY_PATH,
]:
    print(f"- {path}")

inputs
- 07_데이터산출물\m1_compact13_leakage_safe_cv_predictions.csv
- 07_데이터산출물\m1_compact13_leakage_safe_misclassification_audit.csv
- 07_데이터산출물\m1_feature_expansion_features.csv
- 07_데이터산출물\m1_label_window_audit.csv
- 07_데이터산출물\m1_compact_feature_set_summary.csv


## Data

기존 10번 prediction은 이미 group CV out-of-fold 결과다. 여기서는 재학습하지 않고,
normal 35건의 FP/TN 여부와 probability만 집계한다.

In [2]:
def parse_feature_list(value: str) -> list[str]:
    if pd.isna(value) or not str(value).strip():
        return []
    return [item.strip() for item in str(value).split("|") if item.strip()]


predictions_df = pd.read_csv(PREDICTIONS_PATH)
misclassification_df = pd.read_csv(MISCLASSIFICATION_PATH)
features_df = pd.read_csv(FEATURES_PATH)
label_window_audit_df = pd.read_csv(LABEL_WINDOW_AUDIT_PATH)
feature_set_summary_df = pd.read_csv(FEATURE_SET_SUMMARY_PATH)

compact13_features = parse_feature_list(
    feature_set_summary_df.loc[
        feature_set_summary_df["feature_set"].eq("compact13_overlap"), "features"
    ].iloc[0]
)

features_df["source_event_id"] = features_df["source_event_id"].astype(int)
predictions_df["source_event_id"] = predictions_df["source_event_id"].astype(int)
misclassification_df["source_event_id"] = misclassification_df["source_event_id"].astype(int)

normal_features_df = features_df.loc[features_df["label"].eq("normal")].copy()
positive_features_df = features_df.loc[features_df["label"].eq("efd_possible")].copy()
assert len(normal_features_df) == 35
assert len(positive_features_df) == 14
assert len(compact13_features) == 13
assert set(compact13_features).issubset(features_df.columns)
assert set(features_df["manufacturer"].unique()) == {"manufacturer_1"}

normal_window_audit = label_window_audit_df.loc[
    (label_window_audit_df["label"].eq("normal"))
    & (label_window_audit_df["policy"].eq("normal_7d"))
].copy()
assert len(normal_window_audit) == 35
assert normal_window_audit["disturbance_count"].sum() == 0

fault_events = features_df.loc[
    features_df["source_type"].eq("fault_record"), "source_event_id"
].astype(int)
assert not fault_events.isin([20, 34, 69]).any()

print(f"normal rows: {len(normal_features_df)}")
print(f"positive rows: {len(positive_features_df)}")
print(f"compact13 features: {len(compact13_features)}")

normal rows: 35
positive rows: 14
compact13 features: 13


## Hard Normal Audit

`hard_normal`은 main dataset에서 `locked_compact13` 또는 `fold_selected_compact13` 중 하나라도
threshold 0.6 이상으로 FP가 된 normal이다.

In [3]:
def prediction_slice(dataset_id: str, strategy: str) -> pd.DataFrame:
    sliced = predictions_df.loc[
        predictions_df["dataset_id"].eq(dataset_id)
        & predictions_df["feature_strategy"].eq(strategy)
        & predictions_df["model"].eq(MODEL_NAME)
        & predictions_df["label"].eq("normal")
    ].copy()
    assert len(sliced) == 35, f"{dataset_id} {strategy} normal rows={len(sliced)}"
    return sliced[
        [
            "sample_id",
            "source_event_id",
            "fold",
            "y_probability",
            "y_pred",
            "prediction_label",
        ]
    ]


audit_columns = [
    "sample_id",
    "manufacturer",
    "label",
    "label_strength",
    "source_event_id",
    "substation_id",
    "window_start",
    "window_end",
    "window_days",
    "window_policy",
    "sample_count",
    "expected_sample_count",
    "coverage_rate",
    "disturbance_count",
    "disturbance_types",
    "source_type",
]
hard_normal_audit_df = normal_features_df[audit_columns].copy()

for strategy in STRATEGIES:
    strategy_predictions = prediction_slice(MAIN_DATASET, strategy).rename(
        columns={
            "fold": f"{strategy}_fold",
            "y_probability": f"{strategy}_probability",
            "y_pred": f"{strategy}_pred",
            "prediction_label": f"{strategy}_prediction_label",
        }
    )
    hard_normal_audit_df = hard_normal_audit_df.merge(
        strategy_predictions.drop(columns=["source_event_id"]),
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
    hard_normal_audit_df[f"{strategy}_fp"] = (
        hard_normal_audit_df[f"{strategy}_pred"].eq(1)
        & hard_normal_audit_df["label"].eq("normal")
    )

for strategy in COMPACT_STRATEGIES:
    sensitivity_predictions = prediction_slice(SENSITIVITY_DATASET, strategy).rename(
        columns={
            "fold": f"sensitivity_{strategy}_fold",
            "y_probability": f"sensitivity_{strategy}_probability",
            "y_pred": f"sensitivity_{strategy}_pred",
            "prediction_label": f"sensitivity_{strategy}_prediction_label",
        }
    )
    hard_normal_audit_df = hard_normal_audit_df.merge(
        sensitivity_predictions.drop(columns=["source_event_id"]),
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
    hard_normal_audit_df[f"sensitivity_{strategy}_fp"] = (
        hard_normal_audit_df[f"sensitivity_{strategy}_pred"].eq(1)
        & hard_normal_audit_df["label"].eq("normal")
    )

hard_normal_audit_df["compact_fp_count"] = hard_normal_audit_df[
    [f"{strategy}_fp" for strategy in COMPACT_STRATEGIES]
].sum(axis=1)
hard_normal_audit_df["fp_count_across_strategies"] = hard_normal_audit_df[
    [f"{strategy}_fp" for strategy in STRATEGIES]
].sum(axis=1)
hard_normal_audit_df["hard_normal_flag"] = hard_normal_audit_df["compact_fp_count"].gt(0)
hard_normal_audit_df["hard_normal_severity"] = np.select(
    [
        hard_normal_audit_df["compact_fp_count"].eq(2),
        hard_normal_audit_df["compact_fp_count"].eq(1),
    ],
    ["high", "medium"],
    default="none",
)
hard_normal_audit_df["hard_normal_strategy"] = hard_normal_audit_df.apply(
    lambda row: "|".join(
        strategy for strategy in COMPACT_STRATEGIES if bool(row[f"{strategy}_fp"])
    ),
    axis=1,
)
hard_normal_audit_df["max_compact_probability"] = hard_normal_audit_df[
    [f"{strategy}_probability" for strategy in COMPACT_STRATEGIES]
].max(axis=1)
hard_normal_audit_df["sensitivity_compact_fp_count"] = hard_normal_audit_df[
    [f"sensitivity_{strategy}_fp" for strategy in COMPACT_STRATEGIES]
].sum(axis=1)
hard_normal_audit_df["sensitivity_hard_normal_flag"] = hard_normal_audit_df[
    "sensitivity_compact_fp_count"
].gt(0)

hard_normal_audit_df = hard_normal_audit_df.sort_values(
    ["hard_normal_flag", "hard_normal_severity", "max_compact_probability", "source_event_id"],
    ascending=[False, True, False, True],
).reset_index(drop=True)

display(
    hard_normal_audit_df[
        [
            "source_event_id",
            "substation_id",
            "locked_compact13_probability",
            "fold_selected_compact13_probability",
            "expanded154_probability",
            "fp_count_across_strategies",
            "compact_fp_count",
            "hard_normal_flag",
            "hard_normal_severity",
            "hard_normal_strategy",
        ]
    ].head(20)
)
print(hard_normal_audit_df["hard_normal_severity"].value_counts().to_string())

,source_event_id,substation_id,locked_compact13_probability,fold_selected_compact13_probability,expanded154_probability,fp_count_across_strategies,compact_fp_count,hard_normal_flag,hard_normal_severity,hard_normal_strategy
0,35,11,0.440732,0.997498,0.525227,1,1,True,medium,fold_selected_compact13
1,8,6,0.080211,0.955289,0.972570,2,1,True,medium,fold_selected_compact13
2,19,8,0.275242,0.876715,0.905383,2,1,True,medium,fold_selected_compact13
3,39,15,0.092949,0.875113,0.894122,2,1,True,medium,fold_selected_compact13
4,48,11,0.050076,0.775316,0.758369,2,1,True,medium,fold_selected_compact13
5,68,13,0.688434,0.496879,0.929289,2,1,True,medium,locked_compact13
6,33,3,0.514817,0.635146,0.832709,2,1,True,medium,fold_selected_compact13
7,27,12,0.603622,0.458304,0.194086,1,1,True,medium,locked_compact13
8,14,26,0.592358,0.313200,0.490194,0,0,False,none,
9,28,7,0.398082,0.533075,0.199963,0,0,False,none,


hard_normal_severity
none      27
medium     8


## Feature Profile

compact13 feature 평균을 normal 전체, hard normal, true negative normal, positive로 나누어 비교한다.

In [4]:
hard_normal_event_ids = set(
    hard_normal_audit_df.loc[hard_normal_audit_df["hard_normal_flag"], "source_event_id"].astype(int)
)
true_negative_event_ids = set(
    hard_normal_audit_df.loc[~hard_normal_audit_df["hard_normal_flag"], "source_event_id"].astype(int)
)

hard_normal_features_df = normal_features_df.loc[
    normal_features_df["source_event_id"].isin(hard_normal_event_ids)
].copy()
true_negative_features_df = normal_features_df.loc[
    normal_features_df["source_event_id"].isin(true_negative_event_ids)
].copy()

profile_rows = []
all_feature_values = features_df[compact13_features]
overall_mean = all_feature_values.mean()
overall_std = all_feature_values.std(ddof=0).replace(0, np.nan)

for feature in compact13_features:
    all_normal_mean = float(normal_features_df[feature].mean())
    hard_mean = float(hard_normal_features_df[feature].mean()) if len(hard_normal_features_df) else np.nan
    tn_mean = float(true_negative_features_df[feature].mean()) if len(true_negative_features_df) else np.nan
    positive_mean = float(positive_features_df[feature].mean())
    scale = overall_std[feature]
    profile_rows.append(
        {
            "feature": feature,
            "all_normal_mean": all_normal_mean,
            "hard_normal_mean": hard_mean,
            "true_negative_normal_mean": tn_mean,
            "positive_mean": positive_mean,
            "hard_minus_true_negative": hard_mean - tn_mean if pd.notna(hard_mean) and pd.notna(tn_mean) else np.nan,
            "hard_minus_positive": hard_mean - positive_mean if pd.notna(hard_mean) else np.nan,
            "abs_hard_minus_true_negative": abs(hard_mean - tn_mean) if pd.notna(hard_mean) and pd.notna(tn_mean) else np.nan,
            "abs_hard_minus_positive": abs(hard_mean - positive_mean) if pd.notna(hard_mean) else np.nan,
            "hard_normal_z_mean": (hard_mean - overall_mean[feature]) / scale if pd.notna(hard_mean) and pd.notna(scale) else np.nan,
            "true_negative_z_mean": (tn_mean - overall_mean[feature]) / scale if pd.notna(tn_mean) and pd.notna(scale) else np.nan,
            "positive_z_mean": (positive_mean - overall_mean[feature]) / scale if pd.notna(scale) else np.nan,
        }
    )

hard_normal_feature_profile_df = pd.DataFrame(profile_rows).sort_values(
    "abs_hard_minus_true_negative", ascending=False
)
display(hard_normal_feature_profile_df.head(13))

,feature,all_normal_mean,hard_normal_mean,true_negative_normal_mean,positive_mean,hard_minus_true_negative,hard_minus_positive,abs_hard_minus_true_negative,abs_hard_minus_positive,hard_normal_z_mean,true_negative_z_mean,positive_z_mean
5,p_net_meter_flow__last_1d_std_minus_prev_6d_std,-22.384854,-54.178539,-12.964503,-131.571496,-41.214036,77.392957,41.214036,77.392957,-0.004127,0.280540,-0.538684
11,s_hc1_supply_temperature_error__last_minus_first,-1.564214,-6.071250,-0.228796,-5.620000,-5.842454,-0.451250,5.842454,0.451250,-0.370879,0.276280,-0.320894
2,outdoor_temperature__last_minus_first,-1.918929,0.062500,-2.506019,0.188095,2.568519,-0.125595,2.568519,0.125595,0.360561,-0.310813,0.393390
12,s_hc1_supply_temperature_setpoint__last_1d_mea...,1.824638,3.517324,1.323101,-5.361800,2.194222,8.879124,2.194222,8.879124,0.557182,0.230808,-0.763520
8,p_return_gap__last_minus_first,0.481937,-0.506458,0.774795,4.794405,-1.281253,-5.300863,1.281253,5.300863,-0.277657,-0.117448,0.385168
0,outdoor_temperature__last_12h_mean_minus_prev_...,1.369849,0.581253,1.603507,-1.008904,-1.022254,1.590157,1.022254,1.590157,-0.039042,0.327272,-0.608858
9,s_hc1_supply_temperature__last_1d_mean_minus_p...,1.166425,0.488356,1.367334,-4.317134,-0.878978,4.805491,0.878978,4.805491,0.171188,0.340510,-0.754518
1,outdoor_temperature__last_6h_mean_minus_prev_6...,-1.762853,-1.147286,-1.945243,1.215749,0.797958,-2.363035,0.797958,2.363035,-0.096747,-0.424614,0.874183
6,p_net_return_temperature__last_1d_mean_minus_p...,0.535855,0.029524,0.685880,-4.952612,-0.656355,4.982137,0.656355,4.982137,0.212158,0.343304,-0.783319
4,p_hc1_return_temperature__last_1d_mean_minus_p...,0.613896,0.199405,0.736708,-3.560078,-0.537303,3.759483,0.537303,3.759483,0.206036,0.348316,-0.789487


## Substation Summary And Decision Notes

hard normal이 특정 substation에 몰리는지, 그리고 hard normal 평균이 positive 쪽에 더 가까운지 계산한다.

In [5]:
def join_ints(values: pd.Series) -> str:
    return "|".join(str(int(value)) for value in sorted(set(values.astype(int))))


substation_rows = []
for substation_id, group in hard_normal_audit_df.groupby("substation_id", sort=True):
    hard_group = group.loc[group["hard_normal_flag"]]
    substation_rows.append(
        {
            "substation_id": int(substation_id),
            "normal_event_count": len(group),
            "hard_normal_count": int(group["hard_normal_flag"].sum()),
            "high_severity_count": int(group["hard_normal_severity"].eq("high").sum()),
            "medium_severity_count": int(group["hard_normal_severity"].eq("medium").sum()),
            "expanded154_fp_count": int(group["expanded154_fp"].sum()),
            "locked_compact13_fp_count": int(group["locked_compact13_fp"].sum()),
            "fold_selected_compact13_fp_count": int(group["fold_selected_compact13_fp"].sum()),
            "max_compact_probability": float(group["max_compact_probability"].max()),
            "normal_event_ids": join_ints(group["source_event_id"]),
            "hard_normal_event_ids": join_ints(hard_group["source_event_id"]) if len(hard_group) else "",
        }
    )
hard_normal_substation_summary_df = pd.DataFrame(substation_rows).sort_values(
    ["hard_normal_count", "max_compact_probability", "substation_id"],
    ascending=[False, False, True],
)

hard_count = int(hard_normal_audit_df["hard_normal_flag"].sum())
high_count = int(hard_normal_audit_df["hard_normal_severity"].eq("high").sum())
medium_count = int(hard_normal_audit_df["hard_normal_severity"].eq("medium").sum())
expanded_fp_count = int(hard_normal_audit_df["expanded154_fp"].sum())
locked_fp_count = int(hard_normal_audit_df["locked_compact13_fp"].sum())
fold_fp_count = int(hard_normal_audit_df["fold_selected_compact13_fp"].sum())
max_substation_hard_count = int(hard_normal_substation_summary_df["hard_normal_count"].max())
concentrated_substations = hard_normal_substation_summary_df.loc[
    hard_normal_substation_summary_df["hard_normal_count"].ge(2), "substation_id"
].astype(str).tolist()

z_profile = hard_normal_feature_profile_df[
    ["hard_normal_z_mean", "true_negative_z_mean", "positive_z_mean"]
].dropna()
if len(z_profile):
    hard_to_positive_distance = float(
        np.sqrt(((z_profile["hard_normal_z_mean"] - z_profile["positive_z_mean"]) ** 2).mean())
    )
    hard_to_true_negative_distance = float(
        np.sqrt(((z_profile["hard_normal_z_mean"] - z_profile["true_negative_z_mean"]) ** 2).mean())
    )
else:
    hard_to_positive_distance = np.nan
    hard_to_true_negative_distance = np.nan

structurally_similar_to_positive = (
    hard_count >= 5
    and pd.notna(hard_to_positive_distance)
    and pd.notna(hard_to_true_negative_distance)
    and hard_to_positive_distance < hard_to_true_negative_distance * 0.85
)
substation_concentration = max_substation_hard_count >= 2

if structurally_similar_to_positive:
    final_decision = "hard normal이 positive와 구조적으로 너무 비슷해 feature/label 기준 재검토 필요"
    decision_reason = (
        "hard normal의 compact13 평균 패턴이 true negative normal보다 positive 평균에 더 가깝다."
    )
elif substation_concentration:
    final_decision = "특정 substation/window에 FP가 몰려 추가 검토 필요"
    decision_reason = (
        "hard normal이 일부 substation에 2건 이상 몰려 있어 설비별 정상 패턴 차이를 확인해야 한다."
    )
else:
    final_decision = "normal은 그대로 유지하고 hard normal만 audit 태그로 관리"
    decision_reason = (
        "normal window의 disturbance는 0이고, hard normal은 소수 이벤트로 제한된다."
    )

decision_notes = [
    ("final_decision", final_decision, decision_reason),
    ("normal_label_policy", "keep", "회사 제공 normal 라벨은 삭제하거나 재라벨링하지 않는다."),
    ("normal_rows", len(hard_normal_audit_df), "normal 35건이 audit에 모두 포함됐다."),
    ("hard_normal_count", hard_count, "main compact13 계열에서 하나 이상 FP가 난 normal 수다."),
    ("high_severity_count", high_count, "locked와 fold-selected compact13 둘 다 FP인 normal 수다."),
    ("medium_severity_count", medium_count, "둘 중 하나만 FP인 normal 수다."),
    ("expanded154_fp_count", expanded_fp_count, "expanded154에서도 FP였던 normal 수다."),
    ("locked_compact13_fp_count", locked_fp_count, "locked compact13 FP normal 수다."),
    ("fold_selected_compact13_fp_count", fold_fp_count, "fold-selected compact13 FP normal 수다."),
    (
        "max_substation_hard_count",
        max_substation_hard_count,
        "한 substation 안에서 발견된 hard normal 최대 건수다.",
    ),
    (
        "concentrated_substations",
        "|".join(concentrated_substations),
        "hard normal이 2건 이상 나온 substation 목록이다.",
    ),
    (
        "hard_to_positive_z_distance",
        hard_to_positive_distance,
        "compact13 z-score 평균 기준 hard normal과 positive의 거리다.",
    ),
    (
        "hard_to_true_negative_z_distance",
        hard_to_true_negative_distance,
        "compact13 z-score 평균 기준 hard normal과 true negative normal의 거리다.",
    ),
]
hard_normal_decision_notes_df = pd.DataFrame(
    decision_notes, columns=["note_key", "value", "interpretation"]
)

display(hard_normal_substation_summary_df.head(10))
display(hard_normal_decision_notes_df)
print(final_decision)

,substation_id,normal_event_count,hard_normal_count,high_severity_count,medium_severity_count,expanded154_fp_count,locked_compact13_fp_count,fold_selected_compact13_fp_count,max_compact_probability,normal_event_ids,hard_normal_event_ids
9,11,2,2,0,2,1,0,2,0.997498,35|48,35|48
5,6,1,1,0,1,1,0,1,0.955289,8,8
7,8,1,1,0,1,1,0,1,0.876715,19,19
13,15,2,1,0,1,1,0,1,0.875113,4|39,39
11,13,1,1,0,1,1,1,0,0.688434,68,68
2,3,2,1,0,1,1,0,1,0.635146,31|33,33
10,12,1,1,0,1,0,1,0,0.603622,27,27
21,26,1,0,0,0,0,0,0,0.592358,14,
6,7,1,0,0,0,0,0,0,0.533075,28,
14,16,2,0,0,0,0,0,0,0.400719,18|30,


,note_key,value,interpretation
0,final_decision,특정 substation/window에 FP가 몰려 추가 검토 필요,hard normal이 일부 substation에 2건 이상 몰려 있어 설비별 정상...
1,normal_label_policy,keep,회사 제공 normal 라벨은 삭제하거나 재라벨링하지 않는다.
2,normal_rows,35,normal 35건이 audit에 모두 포함됐다.
3,hard_normal_count,8,main compact13 계열에서 하나 이상 FP가 난 normal 수다.
4,high_severity_count,0,locked와 fold-selected compact13 둘 다 FP인 normal...
5,medium_severity_count,8,둘 중 하나만 FP인 normal 수다.
6,expanded154_fp_count,7,expanded154에서도 FP였던 normal 수다.
7,locked_compact13_fp_count,2,locked compact13 FP normal 수다.
8,fold_selected_compact13_fp_count,6,fold-selected compact13 FP normal 수다.
9,max_substation_hard_count,2,한 substation 안에서 발견된 hard normal 최대 건수다.


특정 substation/window에 FP가 몰려 추가 검토 필요


## Visualizations

probability ranking, feature profile, substation 분포를 저장한다.

In [6]:
probability_plot_df = hard_normal_audit_df.sort_values(
    "max_compact_probability", ascending=False
).copy()
labels = [f"E{event}" for event in probability_plot_df["source_event_id"]]
x = np.arange(len(probability_plot_df))
width = 0.36

plt.figure(figsize=(14, 6))
plt.bar(
    x - width / 2,
    probability_plot_df["locked_compact13_probability"],
    width=width,
    label="locked_compact13",
)
plt.bar(
    x + width / 2,
    probability_plot_df["fold_selected_compact13_probability"],
    width=width,
    label="fold_selected_compact13",
)
plt.axhline(THRESHOLD, color="red", linestyle="--", linewidth=1.5, label="threshold 0.6")
plt.xticks(x, labels, rotation=70, ha="right")
plt.ylabel("predicted positive probability")
plt.title("Normal event probability ranking")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FILES["probability_ranking_png"], dpi=160)
plt.close()

heatmap_df = hard_normal_feature_profile_df.set_index("feature")[
    ["true_negative_z_mean", "hard_normal_z_mean", "positive_z_mean"]
].copy()
heatmap_df = heatmap_df.sort_values("hard_normal_z_mean", ascending=False)
plt.figure(figsize=(8, 8))
image = plt.imshow(heatmap_df.values, aspect="auto", cmap="coolwarm")
plt.colorbar(image, label="z-scored group mean")
plt.xticks(
    np.arange(heatmap_df.shape[1]),
    ["true_negative", "hard_normal", "positive"],
    rotation=20,
    ha="right",
)
plt.yticks(
    np.arange(heatmap_df.shape[0]),
    [feature.replace("__", "\n") for feature in heatmap_df.index],
    fontsize=8,
)
plt.title("Compact13 feature profile by group")
plt.tight_layout()
plt.savefig(OUTPUT_FILES["feature_profile_png"], dpi=160)
plt.close()

substation_plot_df = hard_normal_substation_summary_df.loc[
    hard_normal_substation_summary_df["hard_normal_count"].gt(0)
].copy()
if substation_plot_df.empty:
    substation_plot_df = hard_normal_substation_summary_df.head(10).copy()
plt.figure(figsize=(10, 5))
plt.bar(
    substation_plot_df["substation_id"].astype(str),
    substation_plot_df["hard_normal_count"],
    label="hard normal count",
)
plt.scatter(
    substation_plot_df["substation_id"].astype(str),
    substation_plot_df["normal_event_count"],
    color="black",
    marker="o",
    label="normal event count",
)
plt.ylabel("event count")
plt.xlabel("substation_id")
plt.title("Hard normal distribution by substation")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FILES["substation_distribution_png"], dpi=160)
plt.close()

print(OUTPUT_FILES["probability_ranking_png"])
print(OUTPUT_FILES["feature_profile_png"])
print(OUTPUT_FILES["substation_distribution_png"])

07_데이터산출물\m1_hard_normal_probability_ranking.png
07_데이터산출물\m1_hard_normal_feature_profile.png
07_데이터산출물\m1_hard_normal_substation_distribution.png


## Save Outputs And Report

CSV, PNG, Markdown 보고서를 저장한다.

In [7]:
def markdown_table(frame: pd.DataFrame, float_digits: int = 4) -> str:
    display_frame = frame.copy()
    for column in display_frame.columns:
        if pd.api.types.is_float_dtype(display_frame[column]):
            display_frame[column] = display_frame[column].map(
                lambda value: "" if pd.isna(value) else f"{value:.{float_digits}f}"
            )
    display_frame = display_frame.fillna("")

    def format_cell(value) -> str:
        return str(value).replace("|", "\\|").replace("\n", "<br>")

    headers = [format_cell(column) for column in display_frame.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in display_frame.itertuples(index=False):
        lines.append("| " + " | ".join(format_cell(value) for value in row) + " |")
    return "\n".join(lines)


hard_normal_audit_df.to_csv(OUTPUT_FILES["audit"], index=False, encoding="utf-8-sig")
hard_normal_feature_profile_df.to_csv(
    OUTPUT_FILES["feature_profile"], index=False, encoding="utf-8-sig"
)
hard_normal_substation_summary_df.to_csv(
    OUTPUT_FILES["substation_summary"], index=False, encoding="utf-8-sig"
)
hard_normal_decision_notes_df.to_csv(
    OUTPUT_FILES["decision_notes"], index=False, encoding="utf-8-sig"
)

hard_normal_brief = hard_normal_audit_df.loc[
    hard_normal_audit_df["hard_normal_flag"],
    [
        "source_event_id",
        "substation_id",
        "locked_compact13_probability",
        "fold_selected_compact13_probability",
        "expanded154_probability",
        "fp_count_across_strategies",
        "hard_normal_severity",
        "hard_normal_strategy",
    ],
].sort_values(["hard_normal_severity", "source_event_id"])

feature_profile_brief = hard_normal_feature_profile_df[
    [
        "feature",
        "all_normal_mean",
        "hard_normal_mean",
        "true_negative_normal_mean",
        "positive_mean",
        "abs_hard_minus_true_negative",
    ]
].head(8)

substation_brief = hard_normal_substation_summary_df.loc[
    hard_normal_substation_summary_df["hard_normal_count"].gt(0),
    [
        "substation_id",
        "normal_event_count",
        "hard_normal_count",
        "high_severity_count",
        "medium_severity_count",
        "hard_normal_event_ids",
    ],
]

report_parts = [
    "# M1 hard normal false positive 보고서",
    "",
    "## 결론",
    "",
    f"- 최종 판단: {final_decision}",
    "- 회사 제공 normal 라벨은 그대로 유지한다.",
    "- 이번 단계에서는 normal을 삭제하거나 재라벨링하지 않았다.",
    f"- main 기준 hard normal은 {hard_count}건이다.",
    f"- high severity는 {high_count}건, medium severity는 {medium_count}건이다.",
    f"- 판단 근거: {decision_reason}",
    "",
    "## 검증 질문",
    "",
    "`compact13 + threshold 0.6`에서 정상인데 위험처럼 잡히는 normal 이벤트가 어떤 것인지 확인했다.",
    "분석 기준은 기존 10번 out-of-fold prediction이며, 새 모델 학습은 하지 않았다.",
    "",
    "## Hard Normal 정의",
    "",
    "- `locked_compact13` 또는 `fold_selected_compact13` 중 하나라도 threshold 0.6에서 FP이면 `hard_normal=True`로 기록했다.",
    "- 두 compact 전략 모두 FP이면 `high`, 하나만 FP이면 `medium`으로 기록했다.",
    "- `expanded154` FP 여부는 참고로만 함께 남겼다.",
    "",
    "## Hard Normal 목록",
    "",
    markdown_table(hard_normal_brief),
    "",
    "## Feature Profile 상위 차이",
    "",
    markdown_table(feature_profile_brief),
    "",
    "## Substation 분포",
    "",
    markdown_table(substation_brief),
    "",
    "## Decision Notes",
    "",
    markdown_table(hard_normal_decision_notes_df),
    "",
    "## 생성 이미지",
    "",
    "- `07_데이터산출물/m1_hard_normal_probability_ranking.png`",
    "- `07_데이터산출물/m1_hard_normal_feature_profile.png`",
    "- `07_데이터산출물/m1_hard_normal_substation_distribution.png`",
    "",
    "## 검증 결과",
    "",
    "- normal 35건이 audit에 모두 포함됐다.",
    "- normal disturbance count는 0으로 재확인했다.",
    "- Event 20/34/69는 학습/판단 dataset에 없다.",
    "- threshold 0.6 기준 FP/TN 구분은 10번 prediction과 일치한다.",
    "- `hard_normal_flag`와 severity는 정의된 규칙으로 생성됐다.",
    "",
    "## 한계와 주의점",
    "",
    "- 샘플 수가 작아서 hard normal은 제거 대상이 아니라 해석/audit 태그로만 사용해야 한다.",
    "- 이번 분석은 normal 라벨 품질을 부정하는 것이 아니라 모델이 헷갈리는 정상 패턴을 분리한 것이다.",
    "- 모델 파일 저장이나 운영 배포는 하지 않았다.",
    "",
    "## 다음에 볼 것",
    "",
    "- hard normal 이벤트의 센서 시계열을 직접 확인한다.",
    "- 특정 substation에 hard normal이 반복되면 설비별 정상 패턴 보정이 필요한지 확인한다.",
    "- 이후 학습에서는 normal 라벨은 유지하되 hard normal 태그를 평가/해석용 metadata로 둔다.",
    "",
]
REPORT_PATH.write_text("\n".join(report_parts), encoding="utf-8")

print(f"saved report: {REPORT_PATH}")
for name, path in OUTPUT_FILES.items():
    print(f"saved {name}: {path}")

saved report: 07_데이터산출물\11_M1_hard_normal_false_positive_보고서.md
saved audit: 07_데이터산출물\m1_hard_normal_audit.csv
saved feature_profile: 07_데이터산출물\m1_hard_normal_feature_profile.csv
saved substation_summary: 07_데이터산출물\m1_hard_normal_substation_summary.csv
saved decision_notes: 07_데이터산출물\m1_hard_normal_decision_notes.csv
saved probability_ranking_png: 07_데이터산출물\m1_hard_normal_probability_ranking.png
saved feature_profile_png: 07_데이터산출물\m1_hard_normal_feature_profile.png
saved substation_distribution_png: 07_데이터산출물\m1_hard_normal_substation_distribution.png


## Validation

요청한 검증 조건을 실행 결과로 확인한다.

In [8]:
for name, path in OUTPUT_FILES.items():
    assert path.exists(), f"missing output: {name}: {path}"
assert REPORT_PATH.exists(), f"missing report: {REPORT_PATH}"

saved_audit = pd.read_csv(OUTPUT_FILES["audit"])
saved_profile = pd.read_csv(OUTPUT_FILES["feature_profile"])
saved_substation = pd.read_csv(OUTPUT_FILES["substation_summary"])
saved_notes = pd.read_csv(OUTPUT_FILES["decision_notes"])

assert len(saved_audit) == 35
assert saved_audit["label"].eq("normal").all()
assert saved_audit["manufacturer"].eq("manufacturer_1").all()
assert saved_audit["disturbance_count"].sum() == 0
assert not saved_audit["source_event_id"].astype(int).isin([20, 34, 69]).any()
assert len(saved_profile) == 13
assert saved_substation["normal_event_count"].sum() == 35

reconstructed_fp = predictions_df.loc[
    predictions_df["dataset_id"].eq(MAIN_DATASET)
    & predictions_df["model"].eq(MODEL_NAME)
    & predictions_df["label"].eq("normal")
    & predictions_df["feature_strategy"].isin(STRATEGIES),
    ["sample_id", "feature_strategy", "y_pred"],
].copy()
reconstructed_pivot = reconstructed_fp.pivot(
    index="sample_id", columns="feature_strategy", values="y_pred"
).reset_index()
check = saved_audit.merge(reconstructed_pivot, on="sample_id", validate="one_to_one")
for strategy in STRATEGIES:
    assert (
        check[f"{strategy}_fp"].astype(bool).to_numpy()
        == check[strategy].eq(1).to_numpy()
    ).all(), f"FP mismatch for {strategy}"

expected_severity = np.select(
    [saved_audit["compact_fp_count"].eq(2), saved_audit["compact_fp_count"].eq(1)],
    ["high", "medium"],
    default="none",
)
assert (saved_audit["hard_normal_severity"].to_numpy() == expected_severity).all()
assert (
    saved_audit["hard_normal_flag"].to_numpy()
    == saved_audit["compact_fp_count"].gt(0).to_numpy()
).all()

for png_key in [
    "probability_ranking_png",
    "feature_profile_png",
    "substation_distribution_png",
]:
    image = mpimg.imread(OUTPUT_FILES[png_key])
    assert image.size > 0
    assert float(np.nanstd(image)) > 0

text_paths = [
    REPORT_PATH,
    OUTPUT_FILES["audit"],
    OUTPUT_FILES["feature_profile"],
    OUTPUT_FILES["substation_summary"],
    OUTPUT_FILES["decision_notes"],
]
forbidden_terms = ["manufacturer" + "_2", "manufacturer " + "2", "M" + "2"]
for path in text_paths:
    text = path.read_text(encoding="utf-8-sig" if path.suffix == ".csv" else "utf-8")
    for term in forbidden_terms:
        assert term not in text, f"forbidden term {term!r} found in {path}"

print("검증 완료")
print(f"- normal audit rows: {len(saved_audit)}")
print(f"- hard normal rows: {int(saved_audit['hard_normal_flag'].sum())}")
print(f"- high severity rows: {int(saved_audit['hard_normal_severity'].eq('high').sum())}")
print(f"- final decision: {saved_notes.loc[saved_notes['note_key'].eq('final_decision'), 'value'].iloc[0]}")

검증 완료
- normal audit rows: 35
- hard normal rows: 8
- high severity rows: 0
- final decision: 특정 substation/window에 FP가 몰려 추가 검토 필요
